# SSFL on Google Colab

Runs the Semi-Supervised Federated Learning framework on a Colab GPU.

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

Steps: check GPU → get the code → install → run tests → quick sanity run → real experiments → plot results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Get the code

Pick **one** option:
* **A — upload the zip** (default): run the cell and choose `ssfl_framework.zip`.
* **B — GitHub**: set `GITHUB_URL` to the repository URL. For a **private** repo, add a
  GitHub personal access token (read access to the repo's contents) in Colab's 🔑 *Secrets* panel under the name
  `GITHUB_TOKEN` and enable notebook access; it is never printed or saved in the notebook.

Optionally set `USE_DRIVE = True` to keep datasets and results in Google Drive
(they are lost when the Colab runtime resets otherwise).

In [ ]:
GITHUB_URL = ""      # e.g. "https://github.com/<user>/FederatedFramework.git"; leave empty to upload the zip
USE_DRIVE  = False   # True = store data/ and runs/ in Google Drive (MyDrive/ssfl)

import os, shutil, zipfile
os.chdir("/content")
if GITHUB_URL:
    shutil.rmtree("/content/FederatedFramework", ignore_errors=True)
    url = GITHUB_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = GITHUB_URL.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass                                        # public repo / no secret set
    rc = os.system(f"git clone -q {url} /content/FederatedFramework")
    assert rc == 0, "git clone failed - check GITHUB_URL (and GITHUB_TOKEN for private repos)"
else:
    from google.colab import files
    uploaded = files.upload()                       # choose ssfl_framework.zip
    zip_name = next(iter(uploaded))
    shutil.rmtree("/content/FederatedFramework", ignore_errors=True)
    zipfile.ZipFile(zip_name).extractall("/content/FederatedFramework")
os.chdir("/content/FederatedFramework")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/ssfl"
else:
    STORE = "/content/ssfl_store"
DATA_DIR, RUNS_DIR = f"{STORE}/data", f"{STORE}/runs"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)
print(os.listdir("."), "| data ->", DATA_DIR, "| runs ->", RUNS_DIR)

## 2. Install (PyTorch is already on Colab)

In [ ]:
!pip install -q -e ".[gui]" medmnist pytest
!python -m ssfl --list

## Optional: graphical interface
Opens the GUI inside this notebook (and prints a public `gradio.live` link you can open in a new tab).
Set *Output folder* to the `RUNS_DIR` printed above to keep runs in Drive. Stop the cell to close the GUI.

In [ ]:
from ssfl.gui import launch
launch(share=True, debug=True)

## 3. Unit tests (synthetic data, ~1 min)

In [ ]:
!python -m pytest -q tests

## 4. Quick sanity run (synthetic data, seconds)

In [ ]:
!python -m ssfl -c configs/quick_test.yaml --set output_dir={RUNS_DIR}

## 5. Real experiments

`COMMON` applies to every run: Colab's Linux workers (`num_workers=2`) do the augmentation in parallel.
Lower `ROUNDS` for a quick check; the thesis settings are 200 (CIFAR-10) and 100 (MedMNIST).
Each run prints one line per round and writes `metrics.csv` to `RUNS_DIR`.

In [ ]:
ROUNDS = 20
COMMON = f"data.root={DATA_DIR} output_dir={RUNS_DIR} data.num_workers=2 server.rounds={ROUNDS}"

def run(config, name, overrides=""):
    cmd = f"python -m ssfl -c {config} --set name={name} {COMMON} {overrides}"
    print("$", cmd)
    get_ipython().system(cmd)

### 5a. OrganAMNIST — cross-silo (10 clients, C = 1.0), 5% labels

In [ ]:
run("configs/organamnist_fedlabel.yaml", "organ_fedlabel_5", "data.labeled_ratio=0.05")

### 5b. Algorithm comparison on OrganAMNIST
Same data split (same seed) for every algorithm — only `strategy.name` changes.

In [ ]:
for algo in ["supervised", "fixmatch", "fedlabel"]:
    run("configs/organamnist_fedlabel.yaml", f"organ_{algo}_5", f"strategy.name={algo} data.labeled_ratio=0.05")

### 5c. Other thesis settings (uncomment what you need)

In [ ]:
# run("configs/organamnist_fedlabel.yaml", "organ_fedlabel_20", "data.labeled_ratio=0.2")
# run("configs/pathmnist_fedlabel.yaml",   "path_fedlabel_5",   "data.labeled_ratio=0.05")
# run("configs/cifar10_fedlabel.yaml",     "cifar_fedlabel_20", "data.labeled_ratio=0.2")
# run("configs/cifar10_fedlabel.yaml",     "cifar_fedlabel_50", "data.labeled_ratio=0.5")
# Ablations: pseudo-label source and confidence score
# run("configs/organamnist_fedlabel.yaml", "organ_sel_global", "strategy.params.selection=global")
# run("configs/organamnist_fedlabel.yaml", "organ_conf_maxp",  "strategy.params.confidence=max_prob")

## 6. Plot results

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

runs = sorted(glob.glob(f"{RUNS_DIR}/*/metrics.csv"))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
summary = []
for path in runs:
    df = pd.read_csv(path)
    name = os.path.basename(os.path.dirname(path)).rsplit("_", 1)[0]
    if "test_acc" in df:
        d = df.dropna(subset=["test_acc"])
        axes[0].plot(d["round"], d["test_acc"], label=name)
        summary.append({"run": name, "best_acc": d["test_acc"].max(), "final_acc": d["test_acc"].iloc[-1]})
    if "train_mask_rate" in df:
        axes[1].plot(df["round"], df["train_mask_rate"], label=name)
    if "train_pl_acc" in df:
        axes[2].plot(df["round"], df["train_pl_acc"], label=name)
for ax, t in zip(axes, ["Test accuracy", "Pseudo-label mask rate", "Pseudo-label accuracy"]):
    ax.set_title(t); ax.set_xlabel("round"); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
pd.DataFrame(summary).sort_values("best_acc", ascending=False)

## 7. Download results (if not using Drive)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/ssfl_runs", "zip", RUNS_DIR)
files.download("/content/ssfl_runs.zip")